In [5]:
import pandas as pd
import numpy as np

GroupLens实验室（**https://grouplens.org/datasets/movielens**）收集了大量由MovieLens用户提供的从20世纪90年代末到21世纪初的电影评分数据。这些数据包括电影评分、电影元数据（风格类型和年代），以及关于用户的人口统计学数据（年龄、邮编、性别和职业）。基于机器学习算法的推荐系统一般都会对此类数据感兴趣。虽然我不会在本书中详细介绍机器学习技术，但会介绍如何对这种数据进行切片和切块以满足实际需求。

MovieLens1M数据集包含来自6000名用户对4000部电影的100万条评分数据。它分为三个表：评分、用户信息和电影信息。

**原始数据展示**

![jupyter](13.1.png)

In [177]:
# 用户表，ID、性别、年龄、职业、邮政编码
unames = ["user_id", "gender", "age", "occupation", "zip"]  # 指定users中的列名
users = pd.read_table("../datasets/movielens/users.dat", sep="::",
                      header=None, names=unames, engine="python")

# 评分表，ID， 电影ID，评分星级、时间戳
rnames = ["user_id", "movie_id", "rating", "timestamp"]
ratinys = pd.read_table("../datasets/movielens/ratings.dat", sep="::",
                        header=None, names=rnames, engine="python")

# 电影表，电影ID、电影标题、电影体裁
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("../datasets/movielens/movies.dat", sep="::",
                       header=None, names=mnames, engine="python")

users赋给的列名数量不够会产生错误，pd会误解，会将数组中的前几位合并在一起  
此时就是pd将前三列默认为了一行，默认为age列的数值  



![jupyter](13.2.png)    

所以后面解析文件时，列名数量要对的上

In [57]:
# 检查读取的数据：
users.head()

,,age,user_id,occupation
1,F,1,10,48067
2,M,56,16,70072
3,M,25,15,55117
4,M,45,7,02460
5,M,25,20,55455


In [65]:
users.head()

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [68]:
ratinys.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [70]:
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [72]:
ratinys

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


注意，年龄和职业编码为用于表明分组的整数，该数据集的README文件对其做了描述。



分析散布在三个表中的数据可不是一件轻松的事情。   




例如，假设我们想**根据性别和年龄计算某部电影的平均得分。**   
如果将所有数据都合并到一个表中的话，问题就简单多了。  
**先用pandas的merge函数将ratings和users合并到一起，然后再将movies数据也合并进去。**  
**pandas会根据重叠的列名推断出哪些列是合并（或连接）键：**

In [50]:
print(users['user_id'].dtype)  # 检查 'user_id' 列的数据类型
print(ratinys['user_id'].dtype)  # 检查 'user_id' 列的数据类型

object
int64


**此时两个DF内的user_id的类型不同，数据表之间是无法合并的，问题就出在自定义列名对不上数据表格中的内容，本来user_id是对应的第一列，现在对应到第二列的gender，所以数据类型变为了object导致无法合并数据**

In [178]:
# 三个表内连接共十列
data = pd.merge(pd.merge(ratinys, users), movies)
data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


In [84]:
# 按位置输出第一条信息
data.iloc[0]

user_id                                            1
movie_id                                        1193
rating                                             5
timestamp                                  978300760
gender                                             F
age                                                1
occupation                                        10
zip                                            48067
title         One Flew Over the Cuckoo's Nest (1975)
genres                                         Drama
Name: 0, dtype: object

In [100]:
# 按性别计算每部电影的平均得分，使用pivot_table方法：
# 创建一个透视表，行标签为电影名称，列标签为性别。
mean_ratings = data.pivot_table("rating", index="title",  # 通过指定的列和函数计算目标数据
                                columns="gender", aggfunc="mean")

mean_ratings.head()

gender,F,M
title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024


三表合并后的data数据集，通过操作指定的要聚合的列rating，将title设置为行索引，gender设置为列索引，计算每个行索引中列索引的所有值的平均值

通过对三表（users, ratings, movies）合并后的 data 数据集进行操作，pivot_table 函数执行以下操作：

聚合指定的列 rating：这是需要进行聚合计算的列，即电影的评分数据。  
将 title 设为行索引：每个电影标题将作为行索引（index），每一行表示一部电影。  
将 gender 设为列索引：每个性别将作为列索引（columns），通常是 "M"（男性）和 "F"（女性）。  
计算每个行索引中不同性别的平均评分：对每部电影，根据性别进行分组，计算男性观众和女性观众分别给出的平均评分。  

**pivot_table** 是 Pandas 中的一个强大工具，用于基于某些特定的列对数据进行汇总或重塑（类似于 Excel 中的数据透视表）。它可以对数据进行分组、聚合，并生成一个新的表格，以帮助快速理解和分析数据。

In [ ]:
pandas.pivot_table(data, values=None, index=None, columns=None, aggfunc='mean', fill_value=None, margins=False, dropna=True, margins_name='All')

参数：  
data: 输入数据（DataFrame）。  
values: 需要聚合的列，通常是数值列。  
index: 新的行索引，会对这些列进行分组。  
columns: 用作列索引的列，会对这些列进行分组。  
aggfunc: 用于聚合的函数，默认是 'mean'（可以使用 sum, count, max, min, 自定义函数等）。  
fill_value: 用来填充缺失值的数值。  
margins: 是否在输出表的最后添加汇总行/列，总计汇总。  
dropna: 是否丢弃所有缺失值。  
margins_name: 汇总行或列的名称，默认是 'All'。  

In [104]:
# 过滤评论数据不足250条的电影
# 按标题进行分组，利用size()得到包含各电影名称分组大小的Series对象：
ratings_by_title = data.groupby("title").size()  # size统计条数相当于count
ratings_by_title.head()

title
$1,000,000 Duck (1971)            37
'Night Mother (1986)              70
'Til There Was You (1997)         52
'burbs, The (1989)               303
...And Justice for All (1979)    199
dtype: int64

In [106]:
active_titles = ratings_by_title.index[ratings_by_title >= 250]
active_titles

Index([''burbs, The (1989)', '10 Things I Hate About You (1999)',
       '101 Dalmatians (1961)', '101 Dalmatians (1996)', '12 Angry Men (1957)',
       '13th Warrior, The (1999)', '2 Days in the Valley (1996)',
       '20,000 Leagues Under the Sea (1954)', '2001: A Space Odyssey (1968)',
       '2010 (1984)',
       ...
       'X-Men (2000)', 'Year of Living Dangerously (1982)',
       'Yellow Submarine (1968)', 'You've Got Mail (1998)',
       'Young Frankenstein (1974)', 'Young Guns (1988)',
       'Young Guns II (1990)', 'Young Sherlock Holmes (1985)',
       'Zero Effect (1998)', 'eXistenZ (1999)'],
      dtype='object', name='title', length=1216)

In [108]:
# 使用评分数据不少于250条的电影名称索引和.loc，就可以从mean_ratings中选取行了：
mean_ratings = mean_ratings.loc[active_titles]
mean_ratings

gender,F,M
title,,
"'burbs, The (1989)",2.793478,2.962085
10 Things I Hate About You (1999),3.646552,3.311966
101 Dalmatians (1961),3.791444,3.500000
101 Dalmatians (1996),3.240000,2.911215
12 Angry Men (1957),4.184397,4.328421
...,...,...
Young Guns (1988),3.371795,3.425620
Young Guns II (1990),2.934783,2.904025
Young Sherlock Holmes (1985),3.514706,3.363344


In [112]:
# 女性观众最喜欢的电影，以对F列进行降序排列：
too_female_ratings = mean_ratings.sort_values("F", ascending=False)
too_female_ratings.head()

gender,F,M
title,,
"Close Shave, A (1995)",4.644444,4.473795
"Wrong Trousers, The (1993)",4.588235,4.478261
Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.572650,4.464589
Wallace & Gromit: The Best of Aardman Animation (1996),4.563107,4.385075
Schindler's List (1993),4.562602,4.491415


### 计算平均分歧  

**找出男性和女性观众分歧最大的电影。**  
一种办法是给mean_ratings加上一个用于存放平均得分之差的列，并对其进行排序：

In [122]:
# 创建新的列，用于存储评分之间的差距
mean_ratings["diff"] = mean_ratings["M"] - mean_ratings["F"]

# 按"diff"排序即可得到分歧最大的电影，我们来看看女性观众更偏爱哪些电影：
sorted_by_diff = mean_ratings.sort_values("diff")
sorted_by_diff.head()

gender,F,M,diff
title,,,
Dirty Dancing (1987),3.790378,2.959596,-0.830782
Jumpin' Jack Flash (1986),3.254717,2.578358,-0.676359
Grease (1978),3.975265,3.367041,-0.608224
Little Women (1994),3.870588,3.321739,-0.548849
Steel Magnolias (1989),3.901734,3.365957,-0.535777


In [124]:
# 对排序结果逆序则是得到男性喜欢的电影：
sorted_by_diff[::-1].head()

gender,F,M,diff
title,,,
"Good, The Bad and The Ugly, The (1966)",3.494949,4.221300,0.726351
"Kentucky Fried Movie, The (1977)",2.878788,3.555147,0.676359
Dumb & Dumber (1994),2.697987,3.336595,0.638608
"Longest Day, The (1962)",3.411765,4.031447,0.619682
"Cable Guy, The (1996)",2.250000,2.863787,0.613787


切片操作：[start:stop:step]  **[起点:终点:步长]**    

[::1] 正序输出全部数据  
[::-1] 倒叙输出全部数据  
[::-2] 倒叙输出全部数据，步长为2

如果只是想找出**分歧最大的电影**，不考虑性别因素。**分歧可以用方差或标准差测量。**
要这么做，首先计算按照电影名的评分标准差，然后对电影名进行过滤：

In [140]:
# 按title列分组，计算每部电影rating评分之间的标准差
rating_std_by_title = data.groupby("title")["rating"].std()
rating_std_by_title = rating_std_by_title.loc[active_titles]  # 只计算影评大于250人的电影

rating_std_by_title.head()

title
'burbs, The (1989)                   1.107760
10 Things I Hate About You (1999)    0.989815
101 Dalmatians (1961)                0.982103
101 Dalmatians (1996)                1.098717
12 Angry Men (1957)                  0.812731
Name: rating, dtype: float64

In [142]:
# 降序排列并选取前10行，这大概就是分歧最大的10部电影：
rating_std_by_title.sort_values(ascending=False)[:10]

title
Dumb & Dumber (1994)                     1.321333
Blair Witch Project, The (1999)          1.316368
Natural Born Killers (1994)              1.307198
Tank Girl (1995)                         1.277695
Rocky Horror Picture Show, The (1975)    1.260177
Eyes Wide Shut (1999)                    1.259624
Evita (1996)                             1.253631
Billy Madison (1995)                     1.249970
Fear and Loathing in Las Vegas (1998)    1.246408
Bicentennial Man (1999)                  1.245533
Name: rating, dtype: float64

电影分类是以管道分隔（|）字符串形式给出的，因为一部电影可能属于多个分类。  

![jupyter](13.3.png)  

如果按电影分类对评分数据进行分组的话，可以在DataFrame上使用explode方法。  


In [145]:
# 在Series上使用str.split方法将分类字符串分割为分类列表：
movies["genres"].head()

0     Animation|Children's|Comedy
1    Adventure|Children's|Fantasy
2                  Comedy|Romance
3                    Comedy|Drama
4                          Comedy
Name: genres, dtype: object

In [147]:
movies["genres"].head().str.split("|")

0     [Animation, Children's, Comedy]
1    [Adventure, Children's, Fantasy]
2                   [Comedy, Romance]
3                     [Comedy, Drama]
4                            [Comedy]
Name: genres, dtype: object

In [181]:
# 将movie表中体裁列以"|"分隔的形式转化为列表的形式
movies["genre"] = movies.pop("genres").str.split("|")
movies.head()

,movie_id,title,genre
0,1,Toy Story (1995),"[Animation, Children's, Comedy]"
1,2,Jumanji (1995),"[Adventure, Children's, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama]"
4,5,Father of the Bride Part II (1995),[Comedy]


从 movies 数据集中提取 genres 列，将其值通过 str.split("|") 方法按 | 进行拆分，最后将分割后的列表重新赋值回 genres 列。

.pop() 操作是用于提取并移除原始的 genres 列，完成一次性操作。  



**movies.pop("genres")**  
pop() 方法用于提取并删除 movies 数据框中的 genres 列。执行这个操作后，movies 中原来的 genres 列就被移除了，但你可以对其返回的值进行进一步的操作。  
这个返回的值是一个 Series 对象，即 genres 列。  



**.str.split("|")**
str.split("|") 是对 Pandas Series 中的字符串进行操作。它会将 genres 列中的字符串按 | 进行拆分，结果是每个单元格中的字符串被拆分成一个列表。  
例如，"Action|Adventure|Sci-Fi" 将被拆分成 ["Action", "Adventure", "Sci-Fi"]。  



**movies["genres"] = ...**
最后，将拆分后的 genres 列的结果重新赋值给 movies 数据框中的 genres 列。现在，每个 genres 列的单元格都会包含一个由不同类型（genres）组成的列表。

#### 题材分类  



而后，调用movies.explode（"genre"）生成一个新DataFrame，其中的行对应各个电影种类列表的各个“内层”元素。



例如，如果一部电影被分类为既是喜剧也是爱情电影，则结果中就会有两行，一行只是“Comedy”，另一行只是“Romance”：

In [202]:
# 将电影体裁拆分，电影ID为分组
movies_exploded = movies.explode("genre")  # 将列表或数组类型的元素拆分为多行
movies_exploded[:10]

,movie_id,title,genre
0,1,Toy Story (1995),Animation
0,1,Toy Story (1995),Children's
0,1,Toy Story (1995),Comedy
1,2,Jumanji (1995),Adventure
1,2,Jumanji (1995),Children's
1,2,Jumanji (1995),Fantasy
2,3,Grumpier Old Men (1995),Comedy
2,3,Grumpier Old Men (1995),Romance
3,4,Waiting to Exhale (1995),Comedy
3,4,Waiting to Exhale (1995),Drama


In [206]:
# 三表合并，并按照分类进行分组
# 拆分的体裁表、评价表、用户表合并
ratinys_with_genre = pd.merge(pd.merge(movies_exploded, ratinys), users)

ratinys_with_genre.head()

,movie_id,title,genre,user_id,rating,timestamp,gender,age,occupation,zip
0,1,Toy Story (1995),Animation,1,5,978824268,F,1,10,48067
1,1,Toy Story (1995),Animation,6,4,978237008,F,50,9,55117
2,1,Toy Story (1995),Animation,8,4,978233496,M,25,12,11413
3,1,Toy Story (1995),Animation,9,5,978225952,M,25,17,61614
4,1,Toy Story (1995),Animation,10,5,978226474,F,35,1,95370


In [208]:
# 根据电影体裁和观众年龄段进行分组，计算评分的平均数，将age行索引透视为列索引方便对比
genre_ratings = (ratinys_with_genre.groupby(["genre", "age"])["rating"].mean().unstack("age"))
genre_ratings[:10]

age,1,18,25,35,45,50,56
genre,,,,,,,
Action,3.506385,3.447097,3.453358,3.538107,3.528543,3.611333,3.610709
Adventure,3.449975,3.408525,3.443163,3.515291,3.528963,3.628163,3.649064
Animation,3.476113,3.624014,3.701228,3.740545,3.734856,3.780020,3.756233
Children's,3.241642,3.294257,3.426873,3.518423,3.527593,3.556555,3.621822
Comedy,3.497491,3.460417,3.490385,3.561984,3.591789,3.646868,3.650949
Crime,3.710170,3.668054,3.680321,3.733736,3.750661,3.810688,3.832549
Documentary,3.730769,3.865865,3.946690,3.953747,3.966521,3.908108,3.961538
Drama,3.794735,3.721930,3.726428,3.782512,3.784356,3.878415,3.933465
Fantasy,3.317647,3.353778,3.452484,3.482301,3.532468,3.581570,3.532700


groupby(["genre", "age"])["rating"].mean()：按 genre 和 age 分组，计算每组的平均评分。  
.unstack("age")：将 age 列展开为多个列，以电影类型为行索引，年龄段为列索引，形成透视表。